In [1]:
import pandas as pd
from rapidfuzz import process, fuzz
import re

In [2]:
schools = pd.read_csv("DSschools.csv")

In [3]:
schools.head()

,(Do Not Modify) Event registration,(Do Not Modify) Row Checksum,(Do Not Modify) Modified on,School Name,Roll/DENI Number,Teacher Name,Experience,Event Start Date (Event) (Event),Event End Date (Event) (Event),Status,Created on
0,a2692705-2069-ef11-a670-6045bdff2506,6IQPg5ng4U4P3R6xyaSVT1eNDhFBQexv/adu+DMh387Q0n...,13/09/2024 23:30,Colaiste Chraobh Abhann,76076M,Eoin Ryan,Dream Space Academy for Post-Primary,13/09/2024 13:30,13/09/2024 16:30,Approved,02/09/2024 11:39
1,1e33d797-2f66-ef11-a670-7c1e5218d961,EjBftBA8bac8v+eMlNd9yAggUQ3qXqZ5rCiKhJjmrWbe+M...,03/04/2025 23:30,Our Lady’s School,60860Q,Lyndsey Phelan,Dream Space Academy for Post-Primary,03/04/2025 13:30,03/04/2025 16:30,Approved,29/08/2024 17:53
2,1e33ce5a-7265-ef11-a671-0022482b4a0c,KEIGpviSOJq2cZaZezYDB7q8FhynjUXM7fbbZ4hWQduNIl...,07/05/2025 12:30,St.Paul's Secondary School Oughterard,63101K,Diarmuid Lee,Dream Space Academy for Post-Primary,06/05/2025 09:30,06/05/2025 12:30,Approved,28/08/2024 19:18
3,e05cbc58-6765-ef11-a670-000d3a53cdae,wscbLUjlyjGljY+naJmyny2TLUm3JqQ7FT3FIZwKP2ZVVe...,13/12/2024 00:30,Gorey Community School,91492N,Conor Devitt,Dream Space Academy for Post-Primary,12/12/2024 13:30,12/12/2024 16:30,Approved,28/08/2024 18:00
4,7a0020c0-4565-ef11-a670-7c1e521839bc,P1V4xnrTMZBrS0qLnFp5u+VjmgRqjMfvToNooI5UiPKfFX...,11/09/2024 23:30,Roslyn College NLN,40430c,Christine Fitzpatrick,Dream Space Academy for Post-Primary,11/09/2024 13:30,11/09/2024 16:30,Approved,28/08/2024 13:59


In [4]:
schools.shape

(1166, 11)

In [5]:
schools.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1166 entries, 0 to 1165
Data columns (total 11 columns):
 #   Column                              Non-Null Count  Dtype 
---  ------                              --------------  ----- 
 0   (Do Not Modify) Event registration  1166 non-null   object
 1   (Do Not Modify) Row Checksum        1166 non-null   object
 2   (Do Not Modify) Modified on         1166 non-null   object
 3   School Name                         1164 non-null   object
 4   Roll/DENI Number                    869 non-null    object
 5   Teacher Name                        1163 non-null   object
 6   Experience                          1165 non-null   object
 7   Event Start Date (Event) (Event)    1166 non-null   object
 8   Event End Date (Event) (Event)      1166 non-null   object
 9   Status                              1166 non-null   object
 10  Created on                          1166 non-null   object
dtypes: object(11)
memory usage: 100.3+ KB


In [6]:
# Convert the 'Event Start Date (Event) (Event)' column to datetime
schools['Event Start Date (Event) (Event)'] = pd.to_datetime(schools['Event Start Date (Event) (Event)'], dayfirst=True)

# Now sort by this column from oldest to newest
schools = schools.sort_values('Event Start Date (Event) (Event)', ascending=True)

In [7]:
schools = schools.rename(columns={
    'Event Start Date (Event) (Event)': 'Event Start',
    'Event End Date (Event) (Event)': 'Event End'
})

In [8]:
schools = schools.drop(columns=[
    '(Do Not Modify) Event registration',
    '(Do Not Modify) Row Checksum',
    '(Do Not Modify) Modified on',
    'Teacher Name',
    'Status',
    'Created on'
])

In [9]:
schools.shape

(1166, 5)

In [10]:
school_counts = schools['School Name'].value_counts()
duplicates_only = school_counts[school_counts > 1]
print(duplicates_only)

School Name
Scoil Mhuire                       9
Scoil Mhuire GNS                   8
Loreto College Junior School       8
St.Patrick's B.N.S (Hollypark)     8
Mercy Ballymahon                   7
                                  ..
Saint ultans                       2
St. Felim's N.S.                   2
Rush National School               2
Greenlanes NS                      2
Monkstown educate together         2
Name: count, Length: 204, dtype: int64


In [11]:
duplicate_names = schools['School Name'][schools['School Name'].duplicated()].unique()
print(duplicate_names)

['Inspiring the Future' 'Firhouse Community College'
 'Loreto Primary School' 'Stackallen NS' 'Scoil Santain'
 'Colaiste Pobail Acla' 'Delgany National School' "St. Laurence's N.S."
 'Scoil Bhríde' 'Mount Sackville Secondary School' 'Ardgillan College'
 'Saint Ultans ' 'Mercy Ballymahon' 'Monkstown educate together'
 'Colaiste Eanna CBS' 'Naas Community College' 'Gaelscoil Eiscir Riada'
 'Avondale Community College' 'Rosemont School'
 'Adamstown Community College' 'Coláiste Bríde' 'Colaiste Muire'
 'Divine Word NS' 'Scoil Mhuire GNS' "St. Clare's Primary School "
 "St.Patrick's B.N.S (Hollypark) " 'Scoil Assaim BNS' 'Holy Trinity NS'
 'Divine word ns' 'Glenart College' 'Abbey Community College'
 "St. Mary's Secondary School Edenderry" 'Scoil Mhuire'
 'Palmerstown Community School' 'Scoil Mhuire gan Smál'
 'Temple Carrig School' 'Manor House School' "St. Kevin's GNS"
 "St Michael's College" 'Gaelscoil Bhaile Brigín' 'Gaelscoil Moshíológ'
 'Comeragh College' 'Coláiste Eoin' 'Ardgillan Co

In [12]:
# Count how many times each school appears
school_counts = schools['School Name'].value_counts()

# Filter to only those appearing more than 5 times
frequent_duplicates = school_counts[school_counts > 4]
print(frequent_duplicates)

School Name
Scoil Mhuire                                    9
Scoil Mhuire GNS                                8
Loreto College Junior School                    8
St.Patrick's B.N.S (Hollypark)                  8
Mercy Ballymahon                                7
Assumption Secondary School                     6
Gaelscoil Naomh Pádraig                         6
St. Laurence's N.S.                             6
Loreto Primary School                           6
Naas Community College                          6
Adamstown Community College                     6
Firhouse Community College                      5
Glenart College                                 5
Mercy Secondary School Ballymahon               5
grennan college                                 5
Holy Trinity                                    5
St. Kevin's Boys' School                        5
Grange Community College                        5
Colaiste Bride Presentation Secondary School    5
Stackallen NS                         

In [13]:
filtered_schools = schools[schools['School Name'].isin(['Scoil Mhuire', 'Scoil Mhuire GNS'])]
print(filtered_schools)

           School Name Roll/DENI Number                            Experience  \
1111      Scoil Mhuire              NaN       Dream Space Academy for Primary   
1073  Scoil Mhuire GNS              NaN       Dream Space Academy for Primary   
932   Scoil Mhuire GNS              NaN       Dream Space Academy for Primary   
819   Scoil Mhuire GNS           13447Q       Dream Space Academy for Primary   
866       Scoil Mhuire           19987J       Dream Space Academy for Primary   
865       Scoil Mhuire           19987j       Dream Space Academy for Primary   
821   Scoil Mhuire GNS           13447Q       Dream Space Academy for Primary   
820   Scoil Mhuire GNS           13447Q       Dream Space Academy for Primary   
562       Scoil Mhuire           64450R  Dream Space Academy for Post-Primary   
478       Scoil Mhuire           11894I       Dream Space Academy for Primary   
480   Scoil Mhuire GNS           11894i       Dream Space Academy for Primary   
479   Scoil Mhuire GNS      

In [14]:
schools = schools.drop_duplicates(subset=['School Name'], keep='last')

In [15]:
schools.shape

(789, 5)

In [16]:
has_duplicates = schools['School Name'].duplicated().any()
print(has_duplicates)

False


In [17]:
num_missing = schools['Roll/DENI Number'].isna().sum()
print(f"Number of missing Roll/DENI Numbers: {num_missing}")

Number of missing Roll/DENI Numbers: 211


In [18]:
schools.info()

<class 'pandas.core.frame.DataFrame'>
Index: 789 entries, 1134 to 96
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   School Name       788 non-null    object        
 1   Roll/DENI Number  578 non-null    object        
 2   Experience        789 non-null    object        
 3   Event Start       789 non-null    datetime64[ns]
 4   Event End         789 non-null    object        
dtypes: datetime64[ns](1), object(4)
memory usage: 37.0+ KB


In [19]:
PPschools = pd.read_csv("postprimary.csv")

In [20]:
PPschools.head()

,Academic Year,Roll Number,Official School Name,Address 1,Address 2,Address 3,Address 4,Eircode,County,Local Authority,...,School Gender - Post Primary,Irish Classification - Post Primary,Post Primary School Type,Ethos/Religion,FEMALE,MALE,Total 2022-2023,Unnamed: 25,Unnamed: 26,Unnamed: 27
0,2022.0,60010P,Loreto Secondary School,Brick Lane,Balbriggan,Co Dublin,NaN,K32R248,Dublin,Fingal County Council,...,Girls,No subjects taught through Irish,Secondary,CATHOLIC,"1,260",NaN,"1,260",NaN,NaN,NaN
1,2022.0,60021U,St Marys Secondary School,Baldoyle,D13 W208,NaN,NaN,D13W208,Dublin,Fingal County Council,...,Girls,No subjects taught through Irish,Secondary,CATHOLIC,238,NaN,238,NaN,NaN,NaN
2,2022.0,60030V,Blackrock College,Blackrock College,Co Dublin,NaN,NaN,A94FK84,Dublin,Dun Laoghaire Rathdown,...,Boys,No subjects taught through Irish,Secondary,CATHOLIC,NaN,"1,036","1,036",NaN,NaN,NaN
3,2022.0,60040B,Willow Park School,Rock Road,Blackrock,Co Dublin,NaN,A94TW98,Dublin,Dun Laoghaire Rathdown,...,Boys,No subjects taught through Irish,Secondary,CATHOLIC,NaN,216,216,NaN,NaN,NaN
4,2022.0,60041D,Coláiste Eoin,Baile an Bhóthair,Bóthair Stigh Lorgan,Co. Átha Cliath,NaN,A94E122,Dublin,Dun Laoghaire Rathdown,...,Boys,All pupils taught all subjects through Irish,Secondary,CATHOLIC,NaN,496,496,NaN,NaN,NaN


In [21]:
PPschools.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 728 entries, 0 to 727
Data columns (total 28 columns):
 #   Column                               Non-Null Count  Dtype  
---  ------                               --------------  -----  
 0   Academic Year                        727 non-null    float64
 1   Roll Number                          727 non-null    object 
 2   Official School Name                 727 non-null    object 
 3   Address 1                            727 non-null    object 
 4   Address 2                            727 non-null    object 
 5   Address 3                            435 non-null    object 
 6   Address 4                            76 non-null     object 
 7   Eircode                              724 non-null    object 
 8   County                               727 non-null    object 
 9   Local Authority                      727 non-null    object 
 10  Principal Name                       727 non-null    object 
 11  Email                           

In [22]:
PPschools.shape

(728, 28)

In [23]:
primary = pd.read_csv("primary-schools.csv" , skiprows=1)

In [24]:
primary.shape

(3096, 18)

In [25]:
primary.head()

,Roll Number,Official Name,County Description,Local Authority Description,DEIS (Y/N),Gaeltacht Indicator (Y/N),Irish Classification Description,Ethos Description,Junior Infants,Senior Infants,First Class,Second Class,Third Class,Fourth Class,Fifth Class,Sixth Class,Special Class,Total
0,00359V,ST. LOUIS GIRLS NATIONAL SCHOOL,Monaghan,Monaghan County Council,Y,N,No subjects through Irish,Catholic,NaN,NaN,NaN,33,33,41,36,55,NaN,198
1,00373P,DERAVOY NATIONAL SCHOOL,Monaghan,Monaghan County Council,Y,N,No subjects through Irish,Catholic,9,11,9,4,10,11,15,5,NaN,74
2,00467B,BALLINSPITTLE N S,Cork,Cork County Council,N,N,No subjects through Irish,Catholic,23,25,25,19,28,23,33,19,NaN,195
3,00512D,MIDLETON CONVENT N S,Cork,Cork County Council,N,N,No subjects through Irish,Catholic,56,50,28,31,31,29,43,48,28,344
4,00538V,CLOCHAR DAINGEAN,Kerry,Kerry County Council,N,Y,All subjects through Irish,Catholic,22,26,21,16,16,10,8,12,NaN,131


In [26]:
# Rename columns to match for merging (if needed)
PPschools = PPschools.rename(columns={'Roll Number': 'Roll/DENI Number', 'Official School Name': 'School Name'})

In [27]:
# Clean and standardize 'School Name' in both DataFrames
schools['School Name'] = schools['School Name'].str.strip().str.lower()
PPschools['School Name'] = PPschools['School Name'].str.strip().str.lower()


In [28]:
PPschools = PPschools.drop_duplicates(subset='School Name')

In [29]:
schools.head()

,School Name,Roll/DENI Number,Experience,Event Start,Event End
1134,sn muire,NaN,Dream Space Academy for Primary,2018-09-03 10:00:00,03/09/2018 13:00
1131,ballyshannon ns,NaN,Dream Space Academy for Primary,2018-09-04 14:00:00,04/09/2018 17:00
1125,colaiste muire crosshaven,NaN,Dream Space Academy for Post-Primary,2018-09-05 14:00:00,05/09/2018 17:00
1138,newport n.s.,NaN,Dream Space Academy for Primary,2018-09-06 10:00:00,06/09/2018 13:00
1139,st rose's ns,NaN,Dream Space Academy for Primary,2018-09-06 10:00:00,06/09/2018 13:00


In [30]:
schools.shape

(789, 5)

In [31]:
primary['Official Name'] = primary['Official Name'].str.strip().str.lower()


In [32]:
primary = primary.rename(columns={
    'Official Name': 'School Name',
    'Roll Number': 'Roll/DENI Number'
})

In [33]:
def clean_name(name):
    if pd.isna(name):
        return name
    # Remove commas and apostrophes, then normalize whitespace
    name = re.sub(r"[',]", '', name)
    name = re.sub(r'\s+', ' ', name)  # Replace multiple spaces with one
    return name.strip().lower()

# Apply cleaning to all datasets
schools['School Name'] = schools['School Name'].apply(clean_name)
PPschools['School Name'] = PPschools['School Name'].apply(clean_name)
primary['School Name'] = primary['School Name'].apply(clean_name)


In [34]:
# Step 3: Combine PPschools and primary into one reference dataset
all_rolls = pd.concat([PPschools, primary])

In [35]:
all_rolls.head()

,Academic Year,Roll/DENI Number,School Name,Address 1,Address 2,Address 3,Address 4,Eircode,County,Local Authority,...,Junior Infants,Senior Infants,First Class,Second Class,Third Class,Fourth Class,Fifth Class,Sixth Class,Special Class,Total
0,2022.0,60010P,loreto secondary school,Brick Lane,Balbriggan,Co Dublin,NaN,K32R248,Dublin,Fingal County Council,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2022.0,60021U,st marys secondary school,Baldoyle,D13 W208,NaN,NaN,D13W208,Dublin,Fingal County Council,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2022.0,60030V,blackrock college,Blackrock College,Co Dublin,NaN,NaN,A94FK84,Dublin,Dun Laoghaire Rathdown,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2022.0,60040B,willow park school,Rock Road,Blackrock,Co Dublin,NaN,A94TW98,Dublin,Dun Laoghaire Rathdown,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2022.0,60041D,coláiste eoin,Baile an Bhóthair,Bóthair Stigh Lorgan,Co. Átha Cliath,NaN,A94E122,Dublin,Dun Laoghaire Rathdown,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [36]:
all_rolls.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3734 entries, 0 to 3095
Data columns (total 43 columns):
 #   Column                               Non-Null Count  Dtype  
---  ------                               --------------  -----  
 0   Academic Year                        637 non-null    float64
 1   Roll/DENI Number                     3732 non-null   object 
 2   School Name                          3732 non-null   object 
 3   Address 1                            637 non-null    object 
 4   Address 2                            637 non-null    object 
 5   Address 3                            398 non-null    object 
 6   Address 4                            75 non-null     object 
 7   Eircode                              635 non-null    object 
 8   County                               637 non-null    object 
 9   Local Authority                      637 non-null    object 
 10  Principal Name                       637 non-null    object 
 11  Email                              

In [37]:
# Step 4: Merge with the main schools DataFrame
schools = schools.merge(
    all_rolls[['School Name', 'Roll/DENI Number']],
    on='School Name',
    how='left',
    suffixes=('', '_ext')
)

In [38]:
schools.head()

,School Name,Roll/DENI Number,Experience,Event Start,Event End,Roll/DENI Number_ext
0,sn muire,NaN,Dream Space Academy for Primary,2018-09-03 10:00:00,03/09/2018 13:00,NaN
1,ballyshannon ns,NaN,Dream Space Academy for Primary,2018-09-04 14:00:00,04/09/2018 17:00,NaN
2,colaiste muire crosshaven,NaN,Dream Space Academy for Post-Primary,2018-09-05 14:00:00,05/09/2018 17:00,NaN
3,newport n.s.,NaN,Dream Space Academy for Primary,2018-09-06 10:00:00,06/09/2018 13:00,NaN
4,st roses ns,NaN,Dream Space Academy for Primary,2018-09-06 10:00:00,06/09/2018 13:00,NaN


In [39]:
# Step 5: Fill missing Roll/DENI Number values using external data
schools['Roll/DENI Number'] = schools['Roll/DENI Number'].fillna(schools['Roll/DENI Number_ext'])



In [40]:
# Optional: Check remaining missing roll numbers
print("Remaining missing roll numbers:", schools['Roll/DENI Number'].isna().sum())

Remaining missing roll numbers: 163


In [41]:
missing_rolls = schools[schools['Roll/DENI Number'].isna()]
print(missing_rolls[['School Name']].drop_duplicates())

                   School Name
0                     sn muire
1              ballyshannon ns
2    colaiste muire crosshaven
3                 newport n.s.
4                  st roses ns
..                         ...
267            rush youthreach
281         crumlin youthreach
340               santa sabina
342     abbey community school
710                        NaN

[157 rows x 1 columns]


In [42]:
# Drop rows where 'School Name' contains 'youthreach'
schools = schools[~schools['School Name'].str.contains('youthreach', case=False, na=False)]

In [43]:
# Optional: Check remaining missing roll numbers
print("Remaining missing roll numbers:", schools['Roll/DENI Number'].isna().sum())

Remaining missing roll numbers: 159


In [44]:
#pip install rapidfuzz

In [45]:
# STEP 1: Clean names function
def clean_name(name):
    if pd.isna(name):
        return ''
    name = re.sub(r"[',]", '', name)  # remove commas and apostrophes
    name = re.sub(r'\s+', ' ', name)  # normalize spaces
    return name.strip().lower()

# STEP 2: Clean school names
schools['School Name Clean'] = schools['School Name'].apply(clean_name)

# Combine PPschools and primary, clean, drop duplicates
external = pd.concat([PPschools, primary])
external = external.rename(columns={
    'Official School Name': 'School Name',
    'Official Name': 'School Name',
    'Roll Number': 'Roll/DENI Number'
}, errors='ignore')  # ignore if already renamed

external['School Name Clean'] = external['School Name'].apply(clean_name)
external = external.drop_duplicates('School Name Clean')

# STEP 3: Build lookup for matching
external_lookup = dict(zip(external['School Name Clean'], external['Roll/DENI Number']))

# STEP 4: Only match rows still missing roll numbers
missing_mask = schools['Roll/DENI Number'].isna()
missing_schools = schools[missing_mask].copy()

# STEP 5: Perform fuzzy matching
matched_rolls = []
matched_names = []
scores = []

for name in missing_schools['School Name Clean']:
    match, score, _ = process.extractOne(name, external_lookup.keys(), scorer=fuzz.token_sort_ratio)
    if score >= 80:
        matched_rolls.append(external_lookup[match])
        matched_names.append(match)
        scores.append(score)
    else:
        matched_rolls.append(None)
        matched_names.append(None)
        scores.append(score)

# STEP 6: Fill in matches
schools.loc[missing_mask, 'Matched School Name'] = matched_names
schools.loc[missing_mask, 'Match Score'] = scores
schools.loc[missing_mask, 'Roll/DENI Number'] = schools.loc[missing_mask, 'Roll/DENI Number'].combine_first(pd.Series(matched_rolls, index=missing_schools.index))

In [46]:
schools[['School Name', 'Matched School Name', 'Match Score']].dropna(subset=['Matched School Name']).head(10)

,School Name,Matched School Name,Match Score
0,sn muire,sn mhuire,94.117647
1,ballyshannon ns,ballyshannon n s,96.774194
8,our lady of good counsel boys,our lady of good counsel boys n s,93.548387
14,st francis senior school,st francis senior n s,84.444444
17,our lady queen of apostles n.s,our lady queen of apostles,92.857143
25,st damians ns,st damiens ns,92.307692
28,scoil mhuire marino,scoil an choroin mhuire,80.952381
36,st. columba’s college,st columbas college,95.000000
38,colaiste de hide,coláiste de híde,87.500000
39,clooneyquinn ns,clooniquin n s,82.758621


In [47]:
schools[['School Name', 'Matched School Name', 'Match Score']].dropna(subset=['Matched School Name']).shape

(76, 3)

In [48]:
# Optional: Check remaining missing roll numbers
print("Remaining missing roll numbers:", schools['Roll/DENI Number'].isna().sum())

Remaining missing roll numbers: 85


In [49]:
missing_rolls = schools[schools['Roll/DENI Number'].isna()]
print(missing_rolls[['School Name']].drop_duplicates())

                             School Name
2              colaiste muire crosshaven
3                           newport n.s.
4                            st roses ns
5    carrick on shannon community school
6                  shoshanna hayden test
..                                   ...
205                       made up school
264       our ladys of wayside kilternan
265                       loreto foxrock
340                         santa sabina
710                                  NaN

[83 rows x 1 columns]


In [50]:
all_rolls.head()

,Academic Year,Roll/DENI Number,School Name,Address 1,Address 2,Address 3,Address 4,Eircode,County,Local Authority,...,Junior Infants,Senior Infants,First Class,Second Class,Third Class,Fourth Class,Fifth Class,Sixth Class,Special Class,Total
0,2022.0,60010P,loreto secondary school,Brick Lane,Balbriggan,Co Dublin,NaN,K32R248,Dublin,Fingal County Council,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2022.0,60021U,st marys secondary school,Baldoyle,D13 W208,NaN,NaN,D13W208,Dublin,Fingal County Council,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2022.0,60030V,blackrock college,Blackrock College,Co Dublin,NaN,NaN,A94FK84,Dublin,Dun Laoghaire Rathdown,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2022.0,60040B,willow park school,Rock Road,Blackrock,Co Dublin,NaN,A94TW98,Dublin,Dun Laoghaire Rathdown,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2022.0,60041D,coláiste eoin,Baile an Bhóthair,Bóthair Stigh Lorgan,Co. Átha Cliath,NaN,A94E122,Dublin,Dun Laoghaire Rathdown,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [51]:
schools = schools[
    schools['School Name'].notna() &  # keep only non-NaN school names
    ~schools['School Name'].str.contains(r'\b(test|fill)\b', case=False, na=False)  # exclude rows with test or fill
]

C:\Users\a-ldrumm\AppData\Local\Temp\ipykernel_28992\2852600737.py:3: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  ~schools['School Name'].str.contains(r'\b(test|fill)\b', case=False, na=False)  # exclude rows with test or fill


In [52]:
# Optional: Check remaining missing roll numbers
print("Remaining missing roll numbers:", schools['Roll/DENI Number'].isna().sum())

Remaining missing roll numbers: 81


In [53]:
def clean_text(text):
    if pd.isna(text):
        return ''
    text = str(text).lower()
    text = re.sub(r"[',]", '', text)  # remove commas, apostrophes
    text = re.sub(r'\s+', ' ', text)  # normalize spaces
    return text.strip()

# Step 1: Create combined key with cleaned name + town
schools['School Name Clean'] = schools['School Name'].apply(clean_text)
schools['Town Clean'] = schools.get('Address 1', pd.Series(['']*len(schools))).apply(clean_text)
schools['Match Key'] = schools['School Name Clean'] + ' ' + schools['Town Clean']

all_rolls['School Name Clean'] = all_rolls['School Name'].apply(clean_text)
all_rolls['Town Clean'] = all_rolls.get('Address 1', pd.Series(['']*len(all_rolls))).apply(clean_text)
all_rolls['Match Key'] = all_rolls['School Name Clean'] + ' ' + all_rolls['Town Clean']

# Step 2: Build lookup dict for matching
lookup = dict(zip(all_rolls['Match Key'], all_rolls['Roll/DENI Number']))

# Step 3: Find missing roll numbers
missing_mask = schools['Roll/DENI Number'].isna()
missing = schools[missing_mask].copy()

# Step 4: Fuzzy match missing 'Match Key' against lookup keys
matched_rolls = []
matched_keys = []
scores = []

for key in missing['Match Key']:
    match, score, _ = process.extractOne(key, lookup.keys(), scorer=fuzz.token_sort_ratio)
    if score >= 80:  # threshold can be adjusted
        matched_rolls.append(lookup[match])
        matched_keys.append(match)
        scores.append(score)
    else:
        matched_rolls.append(None)
        matched_keys.append(None)
        scores.append(score)

# Step 5: Fill in the roll numbers where matched
schools.loc[missing_mask, 'Matched Key'] = matched_keys
schools.loc[missing_mask, 'Match Score'] = scores
schools.loc[missing_mask, 'Roll/DENI Number'] = schools.loc[missing_mask, 'Roll/DENI Number'].combine_first(
    pd.Series(matched_rolls, index=missing.index)
)


In [54]:
#Optional: Check remaining missing roll numbers
print("Remaining missing roll numbers:", schools['Roll/DENI Number'].isna().sum())

Remaining missing roll numbers: 79


In [55]:
schools.head()[['School Name', 'Roll/DENI Number', 'Matched Key', 'Match Score']]

,School Name,Roll/DENI Number,Matched Key,Match Score
0,sn muire,17841R,NaN,94.117647
1,ballyshannon ns,15456E,NaN,96.774194
2,colaiste muire crosshaven,62200H,colaiste muire crosshaven,100.000000
3,newport n.s.,None,None,75.000000
4,st roses ns,None,None,75.000000


In [56]:
schools[schools['Roll/DENI Number'].isna()].head(20)

,School Name,Roll/DENI Number,Experience,Event Start,Event End,Roll/DENI Number_ext,School Name Clean,Matched School Name,Match Score,Town Clean,Match Key,Matched Key
3,newport n.s.,None,Dream Space Academy for Primary,2018-09-06 10:00:00,06/09/2018 13:00,NaN,newport n.s.,None,75.000000,,newport n.s.,None
4,st roses ns,None,Dream Space Academy for Primary,2018-09-06 10:00:00,06/09/2018 13:00,NaN,st roses ns,None,75.000000,,st roses ns,None
5,carrick on shannon community school,None,Dream Space Academy for Post-Primary,2018-09-06 14:00:00,06/09/2018 17:00,NaN,carrick on shannon community school,None,67.647059,,carrick on shannon community school,None
9,holy child killiney,None,Dream Space Academy for Post-Primary,2018-09-10 14:00:00,10/09/2018 17:00,NaN,holy child killiney,None,62.222222,,holy child killiney,None
10,intermediate school killorglin,None,Dream Space Academy for Post-Primary,2018-09-11 10:00:00,11/09/2018 13:00,NaN,intermediate school killorglin,None,76.666667,,intermediate school killorglin,None
11,tyrrelstown etns,None,Dream Space Academy for Primary,2018-09-11 14:00:00,11/09/2018 17:00,NaN,tyrrelstown etns,None,75.000000,,tyrrelstown etns,None
13,coláiste éamonn rís thomas street wexford,None,Dream Space Academy for Post-Primary,2018-09-13 14:00:00,13/09/2018 17:00,NaN,coláiste éamonn rís thomas street wexford,None,59.740260,,coláiste éamonn rís thomas street wexford,None
15,st. augustine’s ns clontuskert,None,Dream Space Academy for Primary,2018-09-17 10:00:00,17/09/2018 13:00,NaN,st. augustine’s ns clontuskert,None,60.869565,,st. augustine’s ns clontuskert,None
16,inspiring the future project,None,Dream Space Academy for Post-Primary,2018-09-18 10:00:00,18/09/2018 13:00,NaN,inspiring the future project,None,51.724138,,inspiring the future project,None
19,carlanstown ns,None,Dream Space Academy for Primary,2018-09-19 14:00:00,19/09/2018 17:00,NaN,carlanstown ns,None,75.862069,,carlanstown ns,None


In [57]:
schools.loc[schools['School Name'].str.strip().str.lower() == 'newport n.s.', 'Roll/DENI Number'] = '19451O'
schools.loc[schools['School Name'].str.strip().str.lower() == 'st roses ns', 'Roll/DENI Number'] = '20010O'
schools.loc[schools['School Name'].str.strip().str.lower() == 'carrick on shannon community school', 'Roll/DENI Number'] = '91496V'
schools.loc[schools['School Name'].str.strip().str.lower() == 'holy child killiney', 'Roll/DENI Number'] = '60250M'

In [58]:
schools = schools[~schools['School Name'].str.strip().str.lower().eq('inspiring the future project')]
schools = schools[~schools['School Name'].str.strip().str.lower().eq('inspiring the future')]


In [59]:
schools.loc[schools['School Name'].str.strip().str.lower() == 'intermediate school killorglin', 'Roll/DENI Number'] = 'V93NV09'
schools.loc[schools['School Name'].str.strip().str.lower() == 'tyrrelstown etns', 'Roll/DENI Number'] = '20201V'
schools.loc[schools['School Name'].str.strip().str.lower() == 'coláiste éamonn rís thomas street wexford', 'Roll/DENI Number'] = '63640R'

In [60]:
schools.loc[schools['School Name'].str.strip().str.lower() == 'st. augustine’s ns clontuskert', 'Roll/DENI Number'] = '17919F'
schools.loc[schools['School Name'].str.strip().str.lower() == 'carlanstown ns', 'Roll/DENI Number'] = '18132Q'
schools.loc[schools['School Name'].str.strip().str.lower() == 'beneavin college', 'Roll/DENI Number'] = '60511O'
schools.loc[schools['School Name'].str.strip().str.lower() == 'glenstalabbey school', 'Roll/DENI Number'] = '64150F'
schools.loc[schools['School Name'].str.strip().str.lower() == 'scoil san treasa kilshanroe', 'Roll/DENI Number'] = '18445O'
schools.loc[schools['School Name'].str.strip().str.lower() == 'pope john paul ii n.s', 'Roll/DENI Number'] = '19627C'
schools.loc[schools['School Name'].str.strip().str.lower() == 'cbs primary tralee', 'Roll/DENI Number'] = '18247K'
schools.loc[schools['School Name'].str.strip().str.lower() == 'presentation senior mullingar', 'Roll/DENI Number'] = '18212O'
schools.loc[schools['School Name'].str.strip().str.lower() == 'gaelcholaiste cheitinn', 'Roll/DENI Number'] = '72420E'


In [61]:
schools = schools[~schools['School Name'].str.strip().str.lower().eq('language immersion private preschool')]
schools = schools[~schools['School Name'].str.strip().str.lower().eq('pope john paul ii n.s.')]

In [62]:
missing_rolls = schools[schools['Roll/DENI Number'].isna()]
print(missing_rolls[['School Name']].drop_duplicates())

                                           School Name
35                         scoil cholmcille blackstaff
37                                         st marks cs
43                              friends school lisburn
49                            o carolan college nobber
55                          st peters college dunboyne
57                         newbuildings primary school
58                                  mount sion primary
59                              s.n. iorball sionnaigh
61                                          gallen c s
62                              gaelscoil an tseanchaí
63                         sportsreach sallynoggin etb
68                                       gortnor abbey
69                            st marys cbs enniscorthy
72                             st cremins multyfarnham
73                           st. marys college dundalk
76                scoil mhuire beal átha n ghaorthaidh
80                      carrigabruiuse national school
83        

In [63]:
# Extract unique school names with missing Roll/DENI Numbers
missing_school_names = missing_rolls['School Name'].dropna().unique()

# Create a dictionary with school names as keys and None as placeholder values
school_roll_library = {name.strip(): None for name in missing_school_names}

In [64]:
list(school_roll_library.items())[:10]

[('scoil cholmcille blackstaff', None),
 ('st marks cs', None),
 ('friends school lisburn', None),
 ('o carolan college nobber', None),
 ('st peters college dunboyne', None),
 ('newbuildings primary school', None),
 ('mount sion primary', None),
 ('s.n. iorball sionnaigh', None),
 ('gallen c s', None),
 ('gaelscoil an tseanchaí', None)]

In [65]:
# Convert dictionary to DataFrame
school_roll_df = pd.DataFrame(list(school_roll_library.items()), columns=['School Name', 'Roll Number'])

# Save to CSV
school_roll_df.to_csv("school_roll_mapping.csv", index=False)

In [66]:
# Load updated file
updated_rolls = pd.read_csv("school_roll_mapping1.csv")

# Convert to dictionary
updated_school_rolls = dict(zip(updated_rolls['School Name'], updated_rolls['Roll Number']))

In [67]:
# Impute Roll Numbers
schools['Roll/DENI Number'] = schools.apply(
    lambda row: updated_school_rolls.get(row['School Name'], row['Roll/DENI Number']),
    axis=1
)

In [68]:
missing_rolls = schools[schools['Roll/DENI Number'].isna()]
print(missing_rolls[['School Name']].drop_duplicates())

                                 School Name
43                    friends school lisburn
57               newbuildings primary school
62                    gaelscoil an tseanchaí
63               sportsreach sallynoggin etb
110                                 headfort
118  digital excellence cluster of 5 schools
136                            blanaid dwyer
170                        st.patricks b.n.s
172        wicklow montessori primary school
187      digital cluster project - 5 schools
199  nord anglia international school dublin
202                        ballymena academy
203                              shoshanna b
205                           made up school
340                             santa sabina


In [69]:
schools.info()

<class 'pandas.core.frame.DataFrame'>
Index: 856 entries, 0 to 873
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   School Name           856 non-null    object        
 1   Roll/DENI Number      839 non-null    object        
 2   Experience            856 non-null    object        
 3   Event Start           856 non-null    datetime64[ns]
 4   Event End             856 non-null    object        
 5   Roll/DENI Number_ext  335 non-null    object        
 6   School Name Clean     856 non-null    object        
 7   Matched School Name   74 non-null     object        
 8   Match Score           151 non-null    float64       
 9   Town Clean            842 non-null    object        
 10  Match Key             842 non-null    object        
 11  Matched Key           2 non-null      object        
dtypes: datetime64[ns](1), float64(1), object(10)
memory usage: 86.9+ KB


In [70]:
# Drop rows with missing Roll/DENI Number
schools = schools.dropna(subset=['Roll/DENI Number'])

In [71]:
schools = schools.drop(columns=[
    'Matched School Name',
    'Match Score',
    'Town Clean',
    'Match Key',
    'Matched Key',
    'Roll/DENI Number_ext'
])

In [72]:
schools.info()

<class 'pandas.core.frame.DataFrame'>
Index: 839 entries, 0 to 873
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   School Name        839 non-null    object        
 1   Roll/DENI Number   839 non-null    object        
 2   Experience         839 non-null    object        
 3   Event Start        839 non-null    datetime64[ns]
 4   Event End          839 non-null    object        
 5   School Name Clean  839 non-null    object        
dtypes: datetime64[ns](1), object(5)
memory usage: 45.9+ KB


In [73]:
schools.head()

,School Name,Roll/DENI Number,Experience,Event Start,Event End,School Name Clean
0,sn muire,17841R,Dream Space Academy for Primary,2018-09-03 10:00:00,03/09/2018 13:00,sn muire
1,ballyshannon ns,15456E,Dream Space Academy for Primary,2018-09-04 14:00:00,04/09/2018 17:00,ballyshannon ns
2,colaiste muire crosshaven,62200H,Dream Space Academy for Post-Primary,2018-09-05 14:00:00,05/09/2018 17:00,colaiste muire crosshaven
3,newport n.s.,19451O,Dream Space Academy for Primary,2018-09-06 10:00:00,06/09/2018 13:00,newport n.s.
4,st roses ns,20010O,Dream Space Academy for Primary,2018-09-06 10:00:00,06/09/2018 13:00,st roses ns


In [74]:
schools['Roll/DENI Number'] = schools['Roll/DENI Number'].astype(str).str.strip().str.upper()
all_rolls['Roll/DENI Number'] = all_rolls['Roll/DENI Number'].astype(str).str.strip().str.upper()

In [75]:
all_rolls.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3734 entries, 0 to 3095
Data columns (total 46 columns):
 #   Column                               Non-Null Count  Dtype  
---  ------                               --------------  -----  
 0   Academic Year                        637 non-null    float64
 1   Roll/DENI Number                     3734 non-null   object 
 2   School Name                          3732 non-null   object 
 3   Address 1                            637 non-null    object 
 4   Address 2                            637 non-null    object 
 5   Address 3                            398 non-null    object 
 6   Address 4                            75 non-null     object 
 7   Eircode                              635 non-null    object 
 8   County                               637 non-null    object 
 9   Local Authority                      637 non-null    object 
 10  Principal Name                       637 non-null    object 
 11  Email                              

In [76]:
# Create a single combined County column
all_rolls['County Final'] = all_rolls['County'].combine_first(all_rolls['County Description'])

In [77]:
all_rolls['County Final'] = all_rolls['County Final'].str.strip().str.title()

In [78]:
all_rolls = all_rolls.drop(columns=['County', 'County Description'])

In [79]:
# Merge dataframes on Roll/DENI Number, keeping only matching rows in schools
schools_final = schools.merge(
    all_rolls,
    how='left',  # keep all rows from schools
    left_on='Roll/DENI Number',
    right_on='Roll/DENI Number',  # or 'Roll/DENI Number' if they have the same column name
    suffixes=('', '_from_all_rolls')
)


In [80]:

# Extract the year and create a new column
schools_final['Year'] = schools_final['Event Start'].dt.year


In [81]:
schools_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 839 entries, 0 to 838
Data columns (total 51 columns):
 #   Column                               Non-Null Count  Dtype         
---  ------                               --------------  -----         
 0   School Name                          839 non-null    object        
 1   Roll/DENI Number                     839 non-null    object        
 2   Experience                           839 non-null    object        
 3   Event Start                          839 non-null    datetime64[ns]
 4   Event End                            839 non-null    object        
 5   School Name Clean                    839 non-null    object        
 6   Academic Year                        342 non-null    float64       
 7   School Name_from_all_rolls           695 non-null    object        
 8   Address 1                            342 non-null    object        
 9   Address 2                            342 non-null    object        
 10  Address 3     

In [82]:
all_rolls.head()

,Academic Year,Roll/DENI Number,School Name,Address 1,Address 2,Address 3,Address 4,Eircode,Local Authority,Principal Name,...,Third Class,Fourth Class,Fifth Class,Sixth Class,Special Class,Total,School Name Clean,Town Clean,Match Key,County Final
0,2022.0,60010P,loreto secondary school,Brick Lane,Balbriggan,Co Dublin,NaN,K32R248,Fingal County Council,MS. ANN M MCDONOUGH,...,NaN,NaN,NaN,NaN,NaN,NaN,loreto secondary school,brick lane,loreto secondary school brick lane,Dublin
1,2022.0,60021U,st marys secondary school,Baldoyle,D13 W208,NaN,NaN,D13W208,Fingal County Council,MS. EDEL GREENE,...,NaN,NaN,NaN,NaN,NaN,NaN,st marys secondary school,baldoyle,st marys secondary school baldoyle,Dublin
2,2022.0,60030V,blackrock college,Blackrock College,Co Dublin,NaN,NaN,A94FK84,Dun Laoghaire Rathdown,MR. ALAN T MAC GINTY,...,NaN,NaN,NaN,NaN,NaN,NaN,blackrock college,blackrock college,blackrock college blackrock college,Dublin
3,2022.0,60040B,willow park school,Rock Road,Blackrock,Co Dublin,NaN,A94TW98,Dun Laoghaire Rathdown,MR. ALAN JAMES THOMAS ROGAN,...,NaN,NaN,NaN,NaN,NaN,NaN,willow park school,rock road,willow park school rock road,Dublin
4,2022.0,60041D,coláiste eoin,Baile an Bhóthair,Bóthair Stigh Lorgan,Co. Átha Cliath,NaN,A94E122,Dun Laoghaire Rathdown,MR. P.S DE POIRE,...,NaN,NaN,NaN,NaN,NaN,NaN,coláiste eoin,baile an bhóthair,coláiste eoin baile an bhóthair,Dublin


In [83]:
all_rolls[['School Name', 'Eircode']].tail(10)

,School Name,Eircode
3086,radharc na mara national school,NaN
3087,réalt na mara national school,NaN
3088,st. catherine of sienna,NaN
3089,scoil naomh pádraig,NaN
3090,bunscoil na cathrach,NaN
3091,scoil naomh muire eas géitine,NaN
3092,castlebar primary school,NaN
3093,scoil mhuire,NaN
3094,st. josephs national school cree,NaN
3095,NaN,NaN


In [84]:
schools_final[['School Name', 'Eircode']].head(10)

,School Name,Eircode
0,sn muire,NaN
1,ballyshannon ns,NaN
2,colaiste muire crosshaven,P43HC98
3,newport n.s.,NaN
4,st roses ns,NaN
5,carrick on shannon community school,NaN
6,st michaels community college,V15TD30
7,our lady of good counsel boys,NaN
8,holy child killiney,A96XP82
9,intermediate school killorglin,NaN


In [85]:
schools_final['County Final'].isna().sum()

144

In [86]:
schools_final.head()

,School Name,Roll/DENI Number,Experience,Event Start,Event End,School Name Clean,Academic Year,School Name_from_all_rolls,Address 1,Address 2,...,Fourth Class,Fifth Class,Sixth Class,Special Class,Total,School Name Clean_from_all_rolls,Town Clean,Match Key,County Final,Year
0,sn muire,17841R,Dream Space Academy for Primary,2018-09-03 10:00:00,03/09/2018 13:00,sn muire,NaN,sn mhuire,NaN,NaN,...,2,3,2,NaN,17,sn mhuire,,sn mhuire,Wexford,2018
1,ballyshannon ns,15456E,Dream Space Academy for Primary,2018-09-04 14:00:00,04/09/2018 17:00,ballyshannon ns,NaN,ballyshannon n s,NaN,NaN,...,22,13,15,NaN,127,ballyshannon n s,,ballyshannon n s,Kildare,2018
2,colaiste muire crosshaven,62200H,Dream Space Academy for Post-Primary,2018-09-05 14:00:00,05/09/2018 17:00,colaiste muire crosshaven,2022.0,colaiste muire,Crosshaven,Co. Cork,...,NaN,NaN,NaN,NaN,NaN,colaiste muire,crosshaven,colaiste muire crosshaven,Cork,2018
3,newport n.s.,19451O,Dream Space Academy for Primary,2018-09-06 10:00:00,06/09/2018 13:00,newport n.s.,NaN,newport central,NaN,NaN,...,23,31,27,17,237,newport central,,newport central,Mayo,2018
4,st roses ns,20010O,Dream Space Academy for Primary,2018-09-06 10:00:00,06/09/2018 13:00,st roses ns,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2018


In [88]:
schools_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 839 entries, 0 to 838
Data columns (total 51 columns):
 #   Column                               Non-Null Count  Dtype         
---  ------                               --------------  -----         
 0   School Name                          839 non-null    object        
 1   Roll/DENI Number                     839 non-null    object        
 2   Experience                           839 non-null    object        
 3   Event Start                          839 non-null    datetime64[ns]
 4   Event End                            839 non-null    object        
 5   School Name Clean                    839 non-null    object        
 6   Academic Year                        342 non-null    float64       
 7   School Name_from_all_rolls           695 non-null    object        
 8   Address 1                            342 non-null    object        
 9   Address 2                            342 non-null    object        
 10  Address 3     

In [93]:

# Define the columns to keep
columns_to_keep = [
    'School Name',
    'Roll/DENI Number',
    'Experience',
    'Event Start',
    'Event End',
    'School Name Clean',
    'Year'
]

# Create the new DataFrame with only rows where all selected columns are non-null
cleaned_data = schools_final.dropna(subset=columns_to_keep)[columns_to_keep]


In [99]:
cleaned_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 839 entries, 0 to 838
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   School Name        839 non-null    object        
 1   Roll/DENI Number   839 non-null    object        
 2   Experience         839 non-null    object        
 3   Event Start        839 non-null    datetime64[ns]
 4   Event End          839 non-null    object        
 5   School Name Clean  839 non-null    object        
 6   Year               839 non-null    int32         
dtypes: datetime64[ns](1), int32(1), object(5)
memory usage: 42.7+ KB


In [100]:

# Step 1: Extract relevant columns and rename for consistency
primary_county = primary[['Roll/DENI Number', 'County Description']].rename(columns={'County Description': 'county'})
ppschools_county = PPschools[['Roll/DENI Number', 'County']].rename(columns={'County': 'county'})

# Step 2: Merge cleaned_data with primary
merged_data = cleaned_data.merge(primary_county, on='Roll/DENI Number', how='left')

# Step 3: Merge with PPschools, prioritising its county info
merged_data = merged_data.merge(ppschools_county, on='Roll/DENI Number', how='left', suffixes=('_primary', '_pp'))

# Step 4: Create final 'county' column, preferring PPschools data
merged_data['county'] = merged_data['county_pp'].combine_first(merged_data['county_primary'])

# Step 5: Drop helper columns
merged_data = merged_data.drop(columns=['county_primary', 'county_pp'])


In [105]:
PPschools.info()

<class 'pandas.core.frame.DataFrame'>
Index: 638 entries, 0 to 727
Data columns (total 28 columns):
 #   Column                               Non-Null Count  Dtype  
---  ------                               --------------  -----  
 0   Academic Year                        637 non-null    float64
 1   Roll/DENI Number                     637 non-null    object 
 2   School Name                          637 non-null    object 
 3   Address 1                            637 non-null    object 
 4   Address 2                            637 non-null    object 
 5   Address 3                            398 non-null    object 
 6   Address 4                            75 non-null     object 
 7   Eircode                              635 non-null    object 
 8   County                               637 non-null    object 
 9   Local Authority                      637 non-null    object 
 10  Principal Name                       637 non-null    object 
 11  Email                                

In [103]:
merged_data.head(10)

,School Name,Roll/DENI Number,Experience,Event Start,Event End,School Name Clean,Year,county
0,sn muire,17841R,Dream Space Academy for Primary,2018-09-03 10:00:00,03/09/2018 13:00,sn muire,2018,Wexford
1,ballyshannon ns,15456E,Dream Space Academy for Primary,2018-09-04 14:00:00,04/09/2018 17:00,ballyshannon ns,2018,Kildare
2,colaiste muire crosshaven,62200H,Dream Space Academy for Post-Primary,2018-09-05 14:00:00,05/09/2018 17:00,colaiste muire crosshaven,2018,Cork
3,newport n.s.,19451O,Dream Space Academy for Primary,2018-09-06 10:00:00,06/09/2018 13:00,newport n.s.,2018,Mayo
4,st roses ns,20010O,Dream Space Academy for Primary,2018-09-06 10:00:00,06/09/2018 13:00,st roses ns,2018,NaN
5,carrick on shannon community school,91496V,Dream Space Academy for Post-Primary,2018-09-06 14:00:00,06/09/2018 17:00,carrick on shannon community school,2018,NaN
6,st michaels community college,70860W,Dream Space Academy for Post-Primary,2018-09-07 14:00:00,07/09/2018 17:00,st michaels community college,2018,Clare
7,our lady of good counsel boys,19320W,Dream Space Academy for Primary,2018-09-10 10:00:00,10/09/2018 13:00,our lady of good counsel boys,2018,Dublin
8,holy child killiney,60250M,Dream Space Academy for Post-Primary,2018-09-10 14:00:00,10/09/2018 17:00,holy child killiney,2018,Dublin
9,intermediate school killorglin,V93NV09,Dream Space Academy for Post-Primary,2018-09-11 10:00:00,11/09/2018 13:00,intermediate school killorglin,2018,NaN


In [108]:

# Step 1: Extract and rename DEIS columns
primary_deis = primary[['Roll/DENI Number', 'DEIS (Y/N)']].rename(columns={'DEIS (Y/N)': 'DEIS_primary'})
ppschools_deis = PPschools[['Roll/DENI Number', 'DEIS (Y/N)']].rename(columns={'DEIS (Y/N)': 'DEIS_pp'})

# Step 2: Merge with cleaned_data
merged = merged_data.merge(primary_deis, on='Roll/DENI Number', how='left')
merged = merged.merge(ppschools_deis, on='Roll/DENI Number', how='left')

# Step 3: Prioritise PPschools DEIS values
merged['DEIS (Y/N)'] = merged['DEIS_pp'].combine_first(merged['DEIS_primary'])

# Step 4: Drop helper columns
merged = merged.drop(columns=['DEIS_primary', 'DEIS_pp'])


In [109]:
merged.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 839 entries, 0 to 838
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   School Name        839 non-null    object        
 1   Roll/DENI Number   839 non-null    object        
 2   Experience         839 non-null    object        
 3   Event Start        839 non-null    datetime64[ns]
 4   Event End          839 non-null    object        
 5   School Name Clean  839 non-null    object        
 6   Year               839 non-null    int32         
 7   county             695 non-null    object        
 8   DEIS (Y/N)         695 non-null    object        
dtypes: datetime64[ns](1), int32(1), object(7)
memory usage: 55.8+ KB


In [110]:
# Save to CSV
merged.to_csv("final_clean.csv", index=False)